In [ ]:
# ------------------------- IMPORTS -------------------------
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer.noise import NoiseModel
from qiskit_aer import AerSimulator
from qiskit.circuit.random import random_circuit
from qiskit import transpile, QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.converters import circuit_to_dag
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.instruction import Instruction

import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

import re
import os
from datetime import datetime

In [ ]:
# ------------------------- INITIAL SETUP -------------------------
service = QiskitRuntimeService(name="Aniket")
backend = service.backend("ibm_brisbane")
noise_model = NoiseModel.from_backend(backend)

calib_path = "ibm_brisbane_calibrations_2025-04-05T06_27_38Z.csv"

In [ ]:
# ------------------------- FUNCTIONS -------------------------
def extract_active_physical_qubits(transpiled_circ):
    return sorted({
        qubit._index
        for instr in transpiled_circ.data
        for qubit in instr.qubits
    })

def create_virtual_circuit(transpiled_circ, hardware_to_virtual):
    num_virtual_qubits = len(hardware_to_virtual)
    qr_virtual = QuantumRegister(num_virtual_qubits, 'q')
    cr_virtual = ClassicalRegister(num_virtual_qubits, 'c')
    reduced_circ = QuantumCircuit(qr_virtual, cr_virtual)

    for instr in transpiled_circ.data:
        new_qargs = [qr_virtual[hardware_to_virtual[q._index]] for q in instr.qubits]
        new_cargs = [cr_virtual[transpiled_circ.find_bit(c).index] for c in instr.clbits]
        reduced_circ.append(instr.operation, new_qargs, new_cargs)

    return reduced_circ

def simulate_expectation_values(circuit, noise_model=None, shots=10000):
    n_qubits = circuit.num_qubits
    qc = circuit.remove_final_measurements(inplace=False)
    for q in range(n_qubits):
        label = ['I'] * n_qubits
        label[q] = 'Z'
        op = SparsePauliOp.from_list([("".join(label), 1.0)])
        qc.save_expectation_value(op, qubits=list(range(n_qubits)), label=f"z{q}")
    joint_label = ['Z'] * n_qubits
    joint_op = SparsePauliOp.from_list([("".join(joint_label), 1.0)])
    qc.save_expectation_value(joint_op, qubits=list(range(n_qubits)), label="joint_z")
    
    sim = AerSimulator(noise_model=noise_model, method="statevector")
    if noise_model:
        circuit.measure_all(inplace=True)
        result = sim.run(circuit, shots=shots).result()
        counts = result.get_counts()
        z_expect = []
        for q in range(n_qubits):
            z = 0
            for bitstring, count in counts.items():
                bit = bitstring[::-1][q]
                z += count if bit == '0' else -count
            z_expect.append(z / shots)
        joint = 0
        for bitstring, count in counts.items():
            parity = sum(int(bitstring[::-1][q]) for q in range(n_qubits)) % 2
            eigenvalue = (-1) ** parity
            joint += count * eigenvalue
        return z_expect, joint / shots
    else:
        result = sim.run(qc).result()
        z_expect = [result.data(0)[f"z{q}"] for q in range(n_qubits)]
        joint = result.data(0)["joint_z"]
        return z_expect, joint

def extract_dag_data(dag):
    nodes, edges, edge_features, node_to_uid = {'gates': {}, 'input': {}, 'output': {}}, {}, {}, {}
    def idx(bit):
        return getattr(bit, 'index', getattr(bit, '_index', bit[1] if isinstance(bit, tuple) else None))
    for i, node in enumerate(dag.topological_nodes()):
        uid = f"node_{i}"
        node_to_uid[node] = uid
        qargs = [idx(q) for q in getattr(node, 'qargs', [])]
        cargs = [idx(c) for c in getattr(node, 'cargs', [])]
        if hasattr(node, 'op'):
            nodes['gates'][uid] = {'type': node.op.name, 'qargs': qargs, 'cargs': cargs}
        else:
            cat = 'input' if 'In' in type(node).__name__ else 'output'
            nodes[cat][uid] = {'type': type(node).__name__, 'qargs': qargs, 'cargs': cargs}
    for i, (src, dst, wire) in enumerate(dag.edges()):
        eid = f"edge_{i}"
        edges[eid] = (node_to_uid[src], node_to_uid[dst])
        edge_features[eid] = idx(wire)
    return nodes, edges, edge_features

def parse_calibration_csv(file_path, active_qubits, hardware_to_virtual):
    df = pd.read_csv(file_path)
    cols = [
        'T1 (us)', 'T2 (us)', 'Frequency (GHz)', 'Anharmonicity (GHz)',
        'Readout assignment error ', 'Prob meas0 prep1 ',
        'Prob meas1 prep0 ', 'Readout length (ns)'
    ]
    qubit_info = {
        hardware_to_virtual[q]: {col: row[col] for col in cols}
        for _, row in df.iterrows() if (q := row["Qubit"]) in hardware_to_virtual
    }
    gate_errors = {g: {} for g in ["id", "rz", "sx", "x", "ecr"]}
    for _, row in df.iterrows():
        if (q := row["Qubit"]) not in hardware_to_virtual: continue
        vq = hardware_to_virtual[q]
        gate_errors["id"][vq] = row["ID error "]
        gate_errors["rz"][vq] = row["Z-axis rotation (rz) error "]
        gate_errors["sx"][vq] = row["√x (sx) error "]
        gate_errors["x"][vq] = row["Pauli-X error "]
        for conn, val in re.findall(r'(\d+_\d+):([\d\.]+)', str(row["ECR error "])):
            if all(int(x) in hardware_to_virtual for x in conn.split('_')):
                vp = tuple(sorted(hardware_to_virtual[int(x)] for x in conn.split('_')))
                gate_errors["ecr"].setdefault(vp, {})["error"] = float(val)
        for conn, val in re.findall(r'(\d+_\d+):([\d\.]+)', str(row["Gate time (ns)"])):
            if all(int(x) in hardware_to_virtual for x in conn.split('_')):
                vp = tuple(sorted(hardware_to_virtual[int(x)] for x in conn.split('_')))
                gate_errors["ecr"].setdefault(vp, {})["Gate time"] = float(val)
    return qubit_info, gate_errors

def update_nodes_with_errors(nodes, gate_errors):
    for nid, node in nodes["gates"].items():
        t, q = node["type"], node["qargs"]
        if t in {"rz", "sx", "x"} and q:
            node["error"] = gate_errors.get(t, {}).get(q[0], 0.0)
        elif t == "ecr" and len(q) == 2:
            node["error"] = gate_errors["ecr"].get(tuple(sorted(q)), {}).get("error", 0.0)
        else:
            node["error"] = 0.0
    return nodes

def update_edge_features(edge_features, qubit_info):
    return {
        eid: {"qubit_id": q, "qubit_info": qubit_info.get(q, {})}
        for eid, q in edge_features.items()
    }

def build_gnn_data(nodes, edges, edge_features, gate_errors, qubit_info, backend,
                   z_noisy, joint_noisy, z_ideal, joint_ideal, hardware_to_virtual):
    types = sorted(set(backend.configuration().basis_gates + ["measure", "DAGInNode", "DAGOutNode"]))
    all_nodes = {**nodes["gates"], **nodes["input"], **nodes["output"]}
    node_names = list(all_nodes)
    node_to_idx = {name: i for i, name in enumerate(node_names)}

    def one_hot(t, T): return [int(t == s) for s in T]
    def node_features(node):
        return one_hot(node["type"], types) + [
            len(node.get("qargs", [])), 
            len(node.get("cargs", [])), 
            node.get("error", 0.0), 
            0.0
        ]

    X = np.array([node_features(all_nodes[n]) for n in node_names])
    edge_index = np.array([[node_to_idx[s], node_to_idx[d]] for s, d in edges.values()]).T

    keys = [
        'T1 (us)', 'T2 (us)', 'Frequency (GHz)', 'Anharmonicity (GHz)',
        'Readout assignment error ', 'Prob meas0 prep1 ',
        'Prob meas1 prep0 ', 'Readout length (ns)'
    ]
    edge_attr = np.array([
        [feat["qubit_info"].get(k, 0.0) for k in keys]
        for feat in edge_features.values()
    ])

    # Use sentinel value -2.0 instead of 0.0 for unused qubits
    sentinel = -2.0
    full_z_noisy = np.full(127, sentinel)
    full_z_ideal = np.full(127, sentinel)
    for hw, vq in hardware_to_virtual.items():
        full_z_noisy[hw] = z_noisy[vq]
        full_z_ideal[hw] = z_ideal[vq]

    mapping_tensor = torch.full((127,), -1, dtype=torch.long)
    for hw, vq in hardware_to_virtual.items():
        mapping_tensor[hw] = vq

    return Data(
        x=torch.tensor(X, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        z_noisy=torch.tensor(full_z_noisy, dtype=torch.float),
        z_ideal=torch.tensor(full_z_ideal, dtype=torch.float),
        joint_noisy=torch.tensor(joint_noisy, dtype=torch.float),
        joint_ideal = torch.tensor(joint_ideal, dtype=torch.float),
        mapping=mapping_tensor
    )


In [ ]:
# ------------------------- GENERATE DATA -------------------------
from tqdm.notebook import tqdm
dataset = []
num_samples = 1000  # Change as needed

for i in tqdm(range(num_samples), desc="Sample Generating"):
    circ = random_circuit(10, 5, max_operands=2, measure=True, seed=i)
    transpiled = transpile(circ, backend=backend)
    active_physical = extract_active_physical_qubits(transpiled)
    hardware_to_virtual = {hw: i for i, hw in enumerate(active_physical)}
    reduced = create_virtual_circuit(transpiled, hardware_to_virtual)
    z_ideal, joint_ideal = simulate_expectation_values(reduced, None)
    z_noisy, joint_noisy = simulate_expectation_values(reduced.copy(), noise_model)
    dag = circuit_to_dag(reduced)
    nodes, edges, edge_features = extract_dag_data(dag)
    qubit_info, gate_errors = parse_calibration_csv(calib_path, active_physical, hardware_to_virtual)
    nodes = update_nodes_with_errors(nodes, gate_errors)
    edge_features = update_edge_features(edge_features, qubit_info)
    data = build_gnn_data(nodes, edges, edge_features, gate_errors, qubit_info, backend,
                          z_noisy, joint_noisy, z_ideal, joint_ideal, hardware_to_virtual)
    dataset.append(data)

In [ ]:
# ------------------------- SAVE DATA -------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"gnn_dataset_{timestamp}.pt"
torch.save(dataset, save_path)
print(f"\n📁 Dataset saved to: {save_path}")
